In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn as nn

import numpy as np
import pandas as pd
import pickle, ast

In [2]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# load pickle artifacts
with open('./ddxplus_mappings.pkl', 'rb') as file:
    mappings = pickle.load(file)

disease_to_idx = mappings['disease_to_idx']
symptom_to_idx = mappings['symptom_to_idx']
num_diseases = len(mappings['all_diseases'])
num_symptoms = len(mappings['all_symptoms'])

class DDXPlusValidateDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # create vector x
        x_tensor = torch.zeros(num_symptoms, dtype=torch.float32)

        # flip zero to 1.0
        patient_symptoms = ast.literal_eval(row['EVIDENCES'])
        for symptom in patient_symptoms:
            if symptom in symptom_to_idx:
                x_tensor[symptom_to_idx[symptom]] = 1.0

        # its corresponding disease can be found in disease_to_idx mapping
        y_idx = disease_to_idx[row['PATHOLOGY']]
        y_tensor = torch.tensor(y_idx, dtype=torch.long)

        return x_tensor, y_tensor

class DDXPlusClinicalTriage(nn.Module):
    def __init__(self, input_size, num_classes):
        super(DDXPlusClinicalTriage, self).__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        return self.linear(x)

In [3]:
# create instance of DDXPlusValidateDataset class
validate_csv_path = '../data/raw/release_test_patients.csv'

# instantiate dataset with validate dataset csv file
dataset = DDXPlusValidateDataset(validate_csv_path)

# initialise dataloader for pyTorch
dataloader = DataLoader(dataset, batch_size=1024, shuffle=False)

# initialise model
model = DDXPlusClinicalTriage(num_symptoms, num_diseases).to(device)

# just load the mode instead of initialising loss and adam optimizers.
model.load_state_dict(torch.load('./ddxplus_model.pth'))

# freeze weights
model.eval()

correct_predictions = 0
total_patients = 0

print("🩺 Running Validation Loop...")

with torch.no_grad():
    for batch_X, batch_y in dataloader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # get raw scores
        predictions = model(batch_X)

        _, predicted_indices = torch.max(predictions, 1)

        total_patients += batch_y.size(0)
        correct_predictions += (predicted_indices == batch_y).sum().item()

accuracy = (correct_predictions / total_patients) * 100
print(f"✅ Total Unseen Patients Evaluated: {total_patients}")
print(f"🎯 Real-World Accuracy: {accuracy:.2f}%")

🩺 Running Validation Loop...
✅ Total Unseen Patients Evaluated: 134529
🎯 Real-World Accuracy: 99.75%
